In [1]:
!pip install catboost
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 41.8 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve

from catboost import CatBoostClassifier
import optuna

from sklearn.metrics import average_precision_score
import lightgbm as lgb

In [6]:
df_airlines = pd.read_csv('airlines.csv')
df_airports = pd.read_csv('airports.csv')
df_daily_airport_stats = pd.read_csv('daily_airport_stats.csv')
df_flights = pd.read_parquet('flights.parquet')

print("airlines:")
display(df_airlines.head())
print("\nairposrts")
display(df_airports.head())
print("\ndaily_airport_stats")
display(df_daily_airport_stats.head())
print("\nflights")
display(df_flights.head())

airlines:


,airline_code,airline,flights,on_time_arrival_rate,cancel_rate,first_flight_date,last_flight_date
0,WN,Southwest Airlines,2319717,0.7682,0.0092,2025-01-01,2026-08-31
1,DL,Delta Air Lines,1721263,0.8036,0.0145,2025-01-01,2026-08-31
2,AA,American Airlines,1641071,0.7412,0.0215,2025-01-01,2026-08-31
3,OO,SkyWest Airlines,1416733,0.7866,0.0158,2025-01-01,2026-08-31
4,UA,United Airlines,1354575,0.7842,0.0099,2025-01-01,2026-08-31



airposrts


,iata_code,icao_code,airport_name,city,state,latitude,longitude,elevation_ft,airport_type,departures,arrivals
0,ORD,KORD,Chicago O'Hare International Airport,"Chicago, IL",IL,41.978600,-87.904800,680,large_airport,573580,573532
1,DEN,KDEN,Denver International Airport,"Denver, CO",CO,39.860027,-104.673792,5431,large_airport,531750,531722
2,ATL,KATL,Hartsfield Jackson Atlanta International Airport,"Atlanta, GA",GA,33.636700,-84.428101,1026,large_airport,525113,525091
3,DFW,KDFW,Dallas Fort Worth International Airport,"Dallas/Fort Worth, TX",TX,32.896801,-97.038002,607,large_airport,525000,524995
4,PHX,KPHX,Phoenix Sky Harbor International Airport,"Phoenix, AZ",AZ,33.435302,-112.005905,1135,large_airport,329807,329795



daily_airport_stats


,flight_date,airport,departures_scheduled,cancelled,diverted,dep_delay_mean,dep_del15_rate,arr_del15_rate,weather_delay_minutes,nas_delay_minutes,carrier_delay_minutes,late_aircraft_delay_minutes,cancel_rate
0,2025-01-01,ABE,6,0,0,-7.5000,0.0000,0.0000,0,0,0,0,0.0
1,2025-01-01,ABI,5,0,0,-5.6000,0.0000,0.0000,0,0,0,0,0.0
2,2025-01-01,ABQ,58,0,0,2.1379,0.1034,0.1034,0,56,43,27,0.0
3,2025-01-01,ABR,1,0,0,-7.0000,0.0000,0.0000,0,0,0,0,0.0
4,2025-01-01,ACT,3,0,0,54.0000,0.3333,0.6667,0,0,196,0,0.0



flights


,flight_date,year,month,day_of_week,airline_code,airline,tail_number,flight_number,origin,origin_city,...,diverted,sched_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2025-01-01,2025,1,3,DL,Delta Air Lines,N113DX,2080,SFO,"San Francisco, CA",...,0,229,202,188,1589,<NA>,<NA>,<NA>,<NA>,<NA>
1,2025-01-01,2025,1,3,AA,American Airlines,N152AA,2509,SFO,"San Francisco, CA",...,0,301,261,236,2296,<NA>,<NA>,<NA>,<NA>,<NA>
2,2025-01-01,2025,1,3,AA,American Airlines,N960NN,2158,SMF,"Sacramento, CA",...,0,249,221,202,1781,<NA>,<NA>,<NA>,<NA>,<NA>
3,2025-01-01,2025,1,3,AS,Alaska Airlines,N562AS,210,FAI,"Fairbanks, AK",...,0,218,200,183,1533,<NA>,<NA>,<NA>,<NA>,<NA>
4,2025-01-01,2025,1,3,AA,American Airlines,N811NN,1677,LAX,"Los Angeles, CA",...,0,250,221,198,1744,<NA>,<NA>,<NA>,<NA>,<NA>
